In [2]:
import sys
sys.path.append("../src")

from supply import compute_od_supply
import polars as pl
from pathlib import Path
import plotly.graph_objects as go

PROCESSED_DIR = Path("../data/processed")

# ── run with MVP defaults (currentOD = 0) ─────────────────────────
df_supply = compute_od_supply(
    processed_dir=PROCESSED_DIR,
    solve_date="2025-12-31",
    current_od_bookings={}
)

print("── Jan 2026 OD Supply Forecast ──")
print(df_supply)
print(f"\nMean OD Supply (Jan 2026):           {df_supply['od_supply'].mean():.2f} legs/day")
print(f"Min  OD Supply:                      {df_supply['od_supply'].min():.2f}")
print(f"Max  OD Supply:                      {df_supply['od_supply'].max():.2f}")
print(f"\nMean raw_leg_supply:                 {df_supply['raw_leg_supply_fc'].mean():.2f}")
print(f"Mean program consumed:               {df_supply['program_live_legs_fc'].mean():.2f}")
print(f"Mean non-program consumed:           {df_supply['non_program_live_legs_fc'].mean():.2f}")

# ── sanity check: supply should be positive always ────────────────
neg = df_supply.filter(pl.col("od_supply") <= 0)
print(f"\nDays with zero/negative supply: {len(neg)} (expect 0 or very few)")

# ── visualise the supply waterfall ───────────────────────────────
dates = df_supply["date"].to_list()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=dates, y=df_supply["raw_leg_supply_fc"].to_list(),
    mode="lines+markers", name="Raw leg supply (ceil)",
    line=dict(color="#6b7280", width=1.5, dash="dot")
))
fig.add_trace(go.Scatter(
    x=dates, y=df_supply["program_live_legs_fc"].to_list(),
    mode="lines", name="Program legs (consumed)",
    line=dict(color="#2563eb", width=1.5)
))
fig.add_trace(go.Scatter(
    x=dates, y=df_supply["non_program_live_legs_fc"].to_list(),
    mode="lines", name="Non-program legs (consumed)",
    line=dict(color="#d97706", width=1.5)
))
fig.add_trace(go.Scatter(
    x=dates, y=df_supply["od_supply"].to_list(),
    mode="lines+markers", name="OD Supply (available)",
    line=dict(color="#16a34a", width=2.5),
    marker=dict(size=4)
))

fig.update_layout(
    title="OD Supply Waterfall — Jan 2026 (NREG US)",
    xaxis_title="Departure Date",
    yaxis_title="Legs",
    width=1200, height=700,
    margin=dict(l=50, r=50, t=50, b=120),
    legend=dict(
        orientation="h", yanchor="top",
        y=-0.15, xanchor="center", x=0.5,
        font=dict(size=10)
    )
)
fig.show()

# ── save ─────────────────────────────────────────────────────────
df_supply.write_parquet(PROCESSED_DIR / "jan2026_od_supply.parquet")
print("\n✅ Saved → data/processed/jan2026_od_supply.parquet")

── Jan 2026 OD Supply Forecast ──
shape: (31, 8)
┌────────────┬───────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┐
│ date       ┆ num_tails ┆ delivered_ ┆ raw_leg_su ┆ program_l ┆ non_progr ┆ current_o ┆ od_supply │
│ ---        ┆ ---       ┆ tails_fc   ┆ pply_fc    ┆ ive_legs_ ┆ am_live_l ┆ d_adjuste ┆ ---       │
│ datetime[μ ┆ i64       ┆ ---        ┆ ---        ┆ fc        ┆ egs_fc    ┆ d         ┆ f64       │
│ s]         ┆           ┆ f64        ┆ f64        ┆ ---       ┆ ---       ┆ ---       ┆           │
│            ┆           ┆            ┆            ┆ f64       ┆ f64       ┆ f64       ┆           │
╞════════════╪═══════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2026-01-01 ┆ 43        ┆ 30.620102  ┆ 32.457309  ┆ 26.86     ┆ 5.89      ┆ 0.0       ┆ 0.0       │
│ 00:00:00   ┆           ┆            ┆            ┆           ┆           ┆           ┆           │
│ 2026-01-02 ┆ 43        ┆ 30.678266  ┆ 32


✅ Saved → data/processed/jan2026_od_supply.parquet
